# 历史订单英文描述聚类分析

## tl;dr

- 384 个品种、3,360 条聚合描述全部完成对账。
- 共形成 573 个描述簇；156 个品种被拆分为多个语义簇。
- 88 个品种只有一条描述，标记为样本不足。
- 识别出 7 条最近邻相似度低于 0.60 的离群描述，适合优先人工复核。


## Context & Methods

### Key Assumptions

- 输入为 Qdrant `recommendation_child_v1` 中的聚合历史描述及现有 BGE-M3 向量。
- 每个 by1 独立聚类，形式和规格只用于解释结果，不参与聚类。
- 使用余弦距离、平均链接层次聚类，并在 1～8 个簇之间自动选择。
- 单条描述不强行聚类；最近邻相似度低于 0.60 标记为离群。

重新生成源结果的命令：

```text
C:\Python314\python.exe tools/cluster_by1_descriptions.py --collection recommendation_child_v1 --output-json results/by1_description_clusters.json
```


## Data

In [1]:
import json
from pathlib import Path
import pandas as pd

root = Path.cwd()
result_path = root / "results" / "by1_description_clusters.json"
payload = json.loads(result_path.read_text(encoding="utf-8"))
variety = pd.DataFrame(payload["variety_summary"])
clusters = pd.DataFrame(payload["cluster_summary"])
detail = pd.DataFrame(payload["detail"])
payload["metadata"]

{'generated_at': '2026-07-14T01:07:57.944763+00:00',
 'source_collection': 'recommendation_child_v1',
 'by1_count': 384,
 'description_count': 3360,
 'cluster_count': 573,
 'insufficient_by1_count': 88,
 'outlier_count': 7}

In [2]:
assert len(variety) == payload["metadata"]["by1_count"] == 384
assert len(detail) == payload["metadata"]["description_count"] == 3360
assert len(clusters) == payload["metadata"]["cluster_count"] == 573
{
    "by1_rows": len(variety),
    "cluster_rows": len(clusters),
    "detail_rows": len(detail),
    "duplicate_point_ids": int(detail["point_id"].duplicated().sum()),
}

{'by1_rows': 384,
 'cluster_rows': 573,
 'detail_rows': 3360,
 'duplicate_point_ids': 0}

## Results

In [3]:
cluster_distribution = (
    variety.groupby("cluster_count", as_index=False)
    .agg(by1_count=("by1", "count"))
    .sort_values("cluster_count")
)
cluster_distribution

,cluster_count,by1_count
0,1,228
1,2,133
2,3,14
3,4,8
4,5,1


In [4]:
top_varieties = variety.sort_values(
    ["description_count", "by1"], ascending=[False, True]
).head(10)
top_varieties[[
    "by1", "description_count", "support_total", "cluster_count",
    "silhouette", "average_cohesion", "outlier_count"
]]

,by1,description_count,support_total,cluster_count,silhouette,average_cohesion,outlier_count
355,XD381X,217,5122,2,0.850285,0.951821,1
291,DH77XSR,170,1278,2,0.799378,0.933605,0
322,XD371X,117,1783,4,0.961525,0.996805,0
266,DH77X,103,1128,2,0.811281,0.960779,0
280,DH77XP,82,483,2,0.764103,0.963089,0
141,D71X4,70,619,4,0.965346,0.995980,1
345,XD371XL,64,211,2,0.845190,0.971584,0
83,D371XLV4,52,146,2,0.928214,0.994378,0
228,D81X4,48,941,2,0.965210,0.998232,0
136,D71X28,46,285,2,0.978261,1.000000,0


In [5]:
outliers = detail.loc[
    detail["is_outlier"],
    [
        "by1", "cluster_id", "description", "representative",
        "form_code", "spec", "nearest_neighbor_similarity"
    ],
].sort_values(["by1", "nearest_neighbor_similarity"])
outliers

,by1,cluster_id,description,representative,form_code,spec,nearest_neighbor_similarity
107,D342XC,2,DI DOUBLE ECCENTRIC FLANGED BUTTERFLY VALVE 4 ...,DI DOUBLE ECCENTRIC FLANGED BUTTERFLY VALVE 4 ...,922,B100,0.373286
871,D71X4,4,BUTTERFLY VALVE DN15 EN558 1 20 CONN,BUTTERFLY VALVE DN15 EN558 1 20 CONN,751,B65-1,0.599067
998,D71XK,1,LD71XK B50,DI WAFER BV DI DISC EPDM SEAT CS LEVER 10 DN250,939,B50,0.443355
1676,DB1321,2,UL FM 3GS EXPOXY GROOVED MECHANICAL TEE 3 X1 8...,UL FM 3GS EXPOXY GROOVED MECHANICAL TEE 3 X1 8...,922,B50,0.434890
1695,DFL,1,MUPP10 ASC B POST PLATE FOR 10,MUPP10 ASC B POST PLATE FOR 10,901,D200,0.486459
1698,DFL,3,POST FLANGE,POST FLANGE,901,D100,0.519518
2904,XD381X,2,暂用,暂用,922,B89,0.466648


## Takeaways

1. 多数品种的描述结构较集中：228 个品种只形成 1 个簇，133 个形成 2 个簇。
2. 156 个品种存在多个语义簇，说明同一品种下包含明显不同的描述模板、连接形式或产品语义，应在推荐时保留多代表描述。
3. `XD381X`、`DH77XSR`、`XD371X` 和 `DH77X` 的描述量最大，适合优先用于后续规则整理和簇级召回验证。
4. 7 条离群描述中包含简写、临时文本和疑似跨品种描述，应人工确认后再决定修正、移除或保留为独立模板。
